# Experiment 4: Cross-Referencing All Three Sources

**Goal:** Prove the full cross-reference pipeline works: MusicBrainz → Discogs + Wikipedia for the same entity. Test with multiple artists to check reliability.

**Key questions:**
- Can we reliably go MusicBrainz → Discogs → Wikipedia for the same entity?
- What percentage of artists have all three links?
- How do we handle entities that exist in one source but not others?
- What does a combined data record look like?

## Setup

In [1]:
import musicbrainzngs
import requests
import json
import time
from urllib.parse import unquote

# MusicBrainz setup
musicbrainzngs.set_useragent("KE-CW2-MusicHistory", "0.1", "lucas@example.com")

# Discogs setup
DISCOGS_BASE = "https://api.discogs.com"

# Wikipedia setup
WIKI_API = "https://en.wikipedia.org/w/api.php"

HEADERS = {"User-Agent": "KE-CW2-MusicHistory/0.1 (lucas@example.com)"}

def discogs_get(endpoint, params=None):
    url = f"{DISCOGS_BASE}{endpoint}"
    resp = requests.get(url, headers=HEADERS, params=params)
    if resp.status_code == 429:
        print("  Rate limited, waiting 60s...")
        time.sleep(60)
        resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()

def wiki_get(params):
    params["format"] = "json"
    resp = requests.get(WIKI_API, params=params, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def pp(data):
    print(json.dumps(data, indent=2, default=str))

## 1. Full Pipeline Function

Given an artist name, go through MusicBrainz → extract Discogs + Wikipedia links → fetch from all three.

In [2]:
def cross_reference_artist(artist_name):
    """Full cross-reference pipeline for a single artist."""
    result = {"name": artist_name, "sources": {}}
    
    # --- Step 1: MusicBrainz ---
    print(f"\n{'='*60}")
    print(f"Processing: {artist_name}")
    print(f"{'='*60}")
    
    search = musicbrainzngs.search_artists(artist=artist_name, limit=1)
    if not search["artist-list"]:
        print(f"  MusicBrainz: NOT FOUND")
        return result
    
    mb_artist = search["artist-list"][0]
    mbid = mb_artist["id"]
    
    # Get full details with URL rels and tags
    time.sleep(1.1)  # MB rate limit
    mb_detail = musicbrainzngs.get_artist_by_id(
        mbid, includes=["url-rels", "tags", "artist-rels"]
    )["artist"]
    
    result["sources"]["musicbrainz"] = {
        "mbid": mbid,
        "name": mb_detail["name"],
        "type": mb_detail.get("type", "N/A"),
        "country": mb_detail.get("country", "N/A"),
        "life_span": mb_detail.get("life-span", {}),
        "tags": [t["name"] for t in sorted(
            mb_detail.get("tag-list", []),
            key=lambda t: int(t.get("count", 0)), reverse=True
        )[:5]],
    }
    print(f"  MusicBrainz: ✓ ({mbid})")
    
    # Extract cross-reference URLs
    url_rels = mb_detail.get("url-relation-list", [])
    discogs_url = None
    wikidata_url = None
    wikipedia_url = None
    
    for rel in url_rels:
        target = rel["target"]
        if rel["type"] == "discogs":
            discogs_url = target
        elif rel["type"] == "wikidata":
            wikidata_url = target
        elif rel["type"] == "wikipedia":
            wikipedia_url = target
    
    result["links"] = {
        "discogs_url": discogs_url,
        "wikidata_url": wikidata_url,
        "wikipedia_url": wikipedia_url,
    }
    
    # --- Step 2: Discogs ---
    if discogs_url:
        try:
            discogs_id = int(discogs_url.rstrip('/').split('/')[-1])
            time.sleep(3)  # Discogs rate limit
            dc = discogs_get(f"/artists/{discogs_id}")
            
            result["sources"]["discogs"] = {
                "id": discogs_id,
                "name": dc.get("name", "N/A"),
                "realname": dc.get("realname", "N/A"),
                "profile_length": len(dc.get("profile", "")),
                "profile_preview": dc.get("profile", "")[:200],
                "members": [m["name"] for m in dc.get("members", [])],
                "groups": [g["name"] for g in dc.get("groups", [])],
            }
            print(f"  Discogs: ✓ (ID: {discogs_id})")
        except Exception as e:
            print(f"  Discogs: FAILED ({e})")
    else:
        print(f"  Discogs: NO LINK in MusicBrainz")
    
    # --- Step 3: Wikipedia ---
    wiki_title = None
    
    # Try path 1: Wikidata → Wikipedia
    if wikidata_url:
        try:
            wikidata_id = wikidata_url.rstrip('/').split('/')[-1]
            wd_resp = requests.get(
                f"https://www.wikidata.org/wiki/Special:EntityData/{wikidata_id}.json",
                headers=HEADERS
            )
            if wd_resp.status_code == 200:
                wd_data = wd_resp.json()
                sitelinks = wd_data["entities"][wikidata_id].get("sitelinks", {})
                wiki_title = sitelinks.get("enwiki", {}).get("title")
        except Exception:
            pass
    
    # Try path 2: Wikipedia URL directly from MB
    if not wiki_title and wikipedia_url:
        wiki_title = unquote(wikipedia_url.split("/wiki/")[-1].replace("_", " "))
    
    # Try path 3: just use the artist name
    if not wiki_title:
        wiki_title = artist_name
    
    try:
        wiki_result = wiki_get({
            "action": "query",
            "titles": wiki_title,
            "prop": "extracts|categories",
            "explaintext": True,
            "exintro": True,
            "exlimit": 1,
            "cllimit": 20
        })
        wiki_page = list(wiki_result["query"]["pages"].values())[0]
        extract = wiki_page.get("extract", "")
        categories = [c["title"] for c in wiki_page.get("categories", [])]
        
        result["sources"]["wikipedia"] = {
            "title": wiki_page.get("title", "N/A"),
            "page_id": wiki_page.get("pageid", "N/A"),
            "text_length": len(extract),
            "text_preview": extract[:200],
            "categories_count": len(categories),
            "sample_categories": categories[:5],
        }
        print(f"  Wikipedia: ✓ ('{wiki_title}', {len(extract)} chars)")
    except Exception as e:
        print(f"  Wikipedia: FAILED ({e})")
    
    # --- Summary ---
    sources_found = list(result["sources"].keys())
    result["coverage"] = len(sources_found)
    print(f"  Coverage: {len(sources_found)}/3 ({', '.join(sources_found)})")
    
    return result

## 2. Test with Multiple Artists

Test a diverse set: solo artist, band, classical composer, producer, lesser-known artist.

In [4]:
test_artists = [
    "David Bowie",       # Solo artist, well-known
    "The Beatles",       # Band, very well-known
    "Miles Davis",       # Jazz, different genre
    "Nirvana",           # Grunge band
    "Ludwig van Beethoven",  # Classical composer
]

results = []
for artist in test_artists:
    r = cross_reference_artist(artist)
    results.append(r)
    time.sleep(2)  # Be polite to APIs


Processing: David Bowie
  MusicBrainz: ✓ (5441c29d-3602-4898-b1a1-b77fa23b8e50)
  Discogs: ✓ (ID: 10263)
  Wikipedia: ✓ ('David Bowie', 3051 chars)
  Coverage: 3/3 (musicbrainz, discogs, wikipedia)

Processing: The Beatles
  MusicBrainz: ✓ (b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d)
  Discogs: ✓ (ID: 82730)
  Wikipedia: ✓ ('The Beatles', 3949 chars)
  Coverage: 3/3 (musicbrainz, discogs, wikipedia)

Processing: Miles Davis
  MusicBrainz: ✓ (561d854a-6a28-4aa7-8c99-323e6ce46c2a)
  Discogs: ✓ (ID: 23755)
  Wikipedia: ✓ ('Miles Davis', 3930 chars)
  Coverage: 3/3 (musicbrainz, discogs, wikipedia)

Processing: Nirvana
  MusicBrainz: ✓ (5b11f4ce-a62d-471e-81fc-a69a8278c7da)
  Discogs: ✓ (ID: 125246)
  Wikipedia: ✓ ('Nirvana (band)', 2489 chars)
  Coverage: 3/3 (musicbrainz, discogs, wikipedia)

Processing: Ludwig van Beethoven
  MusicBrainz: ✓ (1f9df192-a621-4f54-8850-2c5373b7eac9)
  Discogs: ✓ (ID: 95544)
  Wikipedia: ✓ ('Ludwig van Beethoven', 3256 chars)
  Coverage: 3/3 (musicbrainz, discogs

## 3. Coverage Summary

How many artists have all three sources?

In [5]:
print(f"{'Artist':<25s} {'MB':>3s} {'DC':>3s} {'WP':>3s} {'Coverage':>10s}")
print("-" * 50)

for r in results:
    sources = r.get("sources", {})
    mb = "✓" if "musicbrainz" in sources else "✗"
    dc = "✓" if "discogs" in sources else "✗"
    wp = "✓" if "wikipedia" in sources else "✗"
    cov = r.get("coverage", 0)
    print(f"{r['name']:<25s} {mb:>3s} {dc:>3s} {wp:>3s} {cov:>8d}/3")

Artist                     MB  DC  WP   Coverage
--------------------------------------------------
David Bowie                 ✓   ✓   ✓        3/3
The Beatles                 ✓   ✓   ✓        3/3
Miles Davis                 ✓   ✓   ✓        3/3
Nirvana                     ✓   ✓   ✓        3/3
Ludwig van Beethoven        ✓   ✓   ✓        3/3


## 4. Combined Data View

What does a merged record look like for one artist?

In [6]:
# Pick the first result (David Bowie) and show the combined data
bowie = results[0]

print("COMBINED RECORD: David Bowie")
print("=" * 60)

mb = bowie["sources"].get("musicbrainz", {})
dc = bowie["sources"].get("discogs", {})
wp = bowie["sources"].get("wikipedia", {})

print(f"\n--- Identifiers ---")
print(f"  MusicBrainz MBID: {mb.get('mbid', 'N/A')}")
print(f"  Discogs ID:       {dc.get('id', 'N/A')}")
print(f"  Wikipedia Page:   {wp.get('page_id', 'N/A')}")
print(f"  Wikidata ID:      {bowie.get('links', {}).get('wikidata_url', 'N/A')}")

print(f"\n--- Basic Info (from MusicBrainz) ---")
print(f"  Name:     {mb.get('name', 'N/A')}")
print(f"  Type:     {mb.get('type', 'N/A')}")
print(f"  Country:  {mb.get('country', 'N/A')}")
ls = mb.get('life_span', {})
print(f"  Born:     {ls.get('begin', 'N/A')}")
print(f"  Died:     {ls.get('end', 'N/A')}")

print(f"\n--- Additional (from Discogs) ---")
print(f"  Real Name: {dc.get('realname', 'N/A')}")
print(f"  Profile:   {dc.get('profile_preview', 'N/A')}")

print(f"\n--- Genres (from MusicBrainz tags) ---")
print(f"  {mb.get('tags', [])}")

print(f"\n--- Text Source (from Wikipedia) ---")
print(f"  Article: {wp.get('title', 'N/A')} ({wp.get('text_length', 0)} chars)")
print(f"  Preview: {wp.get('text_preview', 'N/A')}")
print(f"  Categories: {wp.get('sample_categories', [])}")

COMBINED RECORD: David Bowie

--- Identifiers ---
  MusicBrainz MBID: 5441c29d-3602-4898-b1a1-b77fa23b8e50
  Discogs ID:       10263
  Wikipedia Page:   8786
  Wikidata ID:      https://www.wikidata.org/wiki/Q5383

--- Basic Info (from MusicBrainz) ---
  Name:     David Bowie
  Type:     Person
  Country:  GB
  Born:     1947-01-08
  Died:     2016-01-10

--- Additional (from Discogs) ---
  Real Name: David Robert Jones
  Profile:   British pop/rock singer, musician, songwriter, and actor.

Born: 8 January 1947 in Brixton, London, England, UK.
Died: 10 January 2016 in Manhattan, New York City, USA (aged 69).

Bowie is recogn

--- Genres (from MusicBrainz tags) ---
  ['art rock', 'glam rock', 'alternative rock', 'pop', 'art pop']

--- Text Source (from Wikipedia) ---
  Article: David Bowie (3051 chars)
  Preview: David Robert Jones (8 January 1947 – 10 January 2016), known as David Bowie, was an English singer, songwriter and actor. Regarded as among the most influential musicians of th

## 5. Link Analysis

What cross-reference links did each artist have in MusicBrainz?

In [7]:
print(f"{'Artist':<25s} {'Discogs URL':>12s} {'Wikidata':>10s} {'Wikipedia':>10s}")
print("-" * 60)

for r in results:
    links = r.get("links", {})
    dc = "✓" if links.get("discogs_url") else "✗"
    wd = "✓" if links.get("wikidata_url") else "✗"
    wp = "✓" if links.get("wikipedia_url") else "✗"
    print(f"{r['name']:<25s} {dc:>12s} {wd:>10s} {wp:>10s}")

print(f"\nNote: Even without a direct Wikipedia URL in MB, we can go via Wikidata.")

Artist                     Discogs URL   Wikidata  Wikipedia
------------------------------------------------------------
David Bowie                          ✓          ✓          ✗
The Beatles                          ✓          ✓          ✗
Miles Davis                          ✓          ✓          ✗
Nirvana                              ✓          ✓          ✗
Ludwig van Beethoven                 ✓          ✓          ✗

Note: Even without a direct Wikipedia URL in MB, we can go via Wikidata.


## 6. Summary & Pipeline Decision

Fill in after running:

| Question | Answer |
|---|---|
| Can we reliably cross-reference? | |
| What % have all 3 sources? | |
| Best cross-ref path to Wikipedia? | |
| Best cross-ref path to Discogs? | |
| What if a link is missing? | |

### Proposed Pipeline Order

```
1. Search MusicBrainz for artist → get MBID
2. Fetch MB detail with url-rels, tags, artist-rels
3. Extract Discogs URL → fetch Discogs (genres/styles, profile)
4. Extract Wikidata URL → resolve to Wikipedia title → fetch article text
5. Combine all data into a unified record
6. Map to ontology → generate RDF triples
```

## Next Steps

- **Experiment 5**: Take the combined data and generate actual RDF triples with rdflib
- **Experiment 6**: Run NER/LLM extraction on the Wikipedia text